<a href="https://colab.research.google.com/github/amendoimcomchocolate5-cloud/vis-o_computacional/blob/main/Copy_of_Untitled2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

# Modelo leve, já treinado para reconhecer objetos comuns
modelo = YOLO("yolo26n.pt")

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

# Caminho correto para a pasta que contém as imagens de teste
pasta_teste = Path("/content/sample_data/cachorros")
extensoes_imagem = {".jpg", ".jpeg", ".png", ".bmp"}

if not pasta_teste.exists():
    print(f"A pasta não foi encontrada: {pasta_teste}")
else:
    fotos = [
        arquivo for arquivo in pasta_teste.iterdir()
        if arquivo.is_file() and arquivo.suffix.lower() in extensoes_imagem
    ]

    if not fotos:
        print("Não encontrei imagens nessa pasta.")
    else:
        print(f"Encontrei {len(fotos)} imagem(ns).")

        for foto in fotos:
            resultados = modelo.predict(
                source=str(foto),
                conf=0.25,
                verbose=False
            )

            # Verificação para evitar IndexError caso a imagem falhe ao ser carregada
            if resultados:
                imagem_anotada = resultados[0].plot()

                # Converte as cores para exibição correta no Colab (BGR para RGB)
                imagem_anotada = imagem_anotada[:, :, ::-1]

                print(f"Resultado: {foto.name}")
                display(Image.fromarray(imagem_anotada))
            else:
                print(f"Erro ao processar ou ler a imagem: {foto.name}")

In [ ]:
# Compilar o código C atualizado com inicialização de Xavier
!gcc -O2 rede_neural.c -lm -o rede_neural
print("Compilação com Xavier concluída com sucesso!")

In [ ]:
# Compilar o código C com a inicialização de Xavier
!gcc -O2 /content/rede_neural.c -lm -o /content/rede_neural
print("Compilação da rede neural com inicialização de Xavier concluída com sucesso!")

In [ ]:
!python /content/preparar_imagens_carros.py \
    /content/dataset_carros \
    /content/dataset_pgm \
    /content/dataset_carros/teste/com_carro/sua_imagem.jpg

In [ ]:
!gcc -O2 /content/reconhecer_carros.c -lm -o /content/reconhecer_carros

!/content/reconhecer_carros \
    /content/dataset_pgm/treino.csv \
    /content/dataset_pgm/teste.pgm

In [ ]:
from pathlib import Path

pastas = [
    "/content/dataset_carros/treino/com_carro",
    "/content/dataset_carros/treino/sem_carro",
    "/content/dataset_carros/teste/com_carro",
    "/content/dataset_carros/teste/sem_carro",
]

for caminho in pastas:
    Path(caminho).mkdir(parents=True, exist_ok=True)
    print(f"Pasta criada: {caminho}")

In [ ]:
from pathlib import Path

raiz = Path("/content/dataset_carros")

for pasta in [
    raiz / "treino/com_carro",
    raiz / "treino/sem_carro",
    raiz / "teste/com_carro",
    raiz / "teste/sem_carro",
]:
    imagens = [
        p for p in pasta.iterdir()
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ]
    print(f"{pasta}: {len(imagens)} imagem(ns)")

In [ ]:
from pathlib import Path

for classe in ["com_carro", "sem_carro"]:
    pasta = Path(f"/content/dataset_carros/treino/{classe}")
    imagens = list(pasta.glob("*"))
    print(f"{classe}: {len(imagens)} arquivo(s)")

In [ ]:
!python /content/preparar_imagens_carros.py \
    /content/dataset_carros \
    /content/dataset_pgm \
    /content/dataset_carros/teste/com_carro/foto_teste.jpg

In [ ]:
!ls /content/dataset_carros/teste/com_carro

In [ ]:
!python /content/preparar_imagens_carros.py \
    /content/dataset_carros \
    /content/dataset_pgm \
    /content/dataset_carros/teste/com_carro/carro_01.jpg

In [ ]:
import os

caminho_script = "/content/preparar_imagens_carros.py"

if os.path.exists(caminho_script):
    print(f"✅ O arquivo {caminho_script} existe!")
else:
    print(f"❌ O arquivo {caminho_script} NÃO foi encontrado no ambiente.")
    print("\nArquivos disponíveis em /content/:")
    for f in os.listdir("/content/"):
        if os.path.isfile(os.path.join("/content/", f)):
            print(f" - {f}")

In [ ]:
%%writefile /content/preparar_imagens_carros.py
import sys
from pathlib import Path
from PIL import Image

def converter_para_pgm_16x16(caminho_entrada, caminho_saida):
    """Carrega uma imagem, converte para tons de cinza, redimensiona para 16x16 e salva em PGM (P2)."""
    try:
        with Image.open(caminho_entrada) as img:
            # Converte para tons de cinza e redimensiona
            img_cinza = img.convert("L").resize((16, 16))

            # Escreve o arquivo no formato PGM P2 (texto plano)
            with open(caminho_saida, "w") as f:
                f.write("P2\n")
                f.write("16 16\n")
                f.write("255\n")

                # Escreve os pixels de 0 a 255
                dados = list(img_cinza.getdata())
                for i, pixel in enumerate(dados):
                    f.write(f"{pixel} ")
                    if (i + 1) % 16 == 0:
                        f.write("\n")
        return True
    except Exception as e:
        print(f"Erro ao converter {caminho_entrada}: {e}")
        return False

def main():
    if len(sys.argv) < 3:
        print("Uso: python preparar_imagens_carros.py <pasta_dataset_carros> <pasta_saida_pgm> [imagem_teste_adicional.jpg]")
        sys.exit(1)

    pasta_carros = Path(sys.argv[1])
    pasta_saida = Path(sys.argv[2])

    pasta_saida.mkdir(parents=True, exist_ok=True)

    csv_linhas = []
    extensoes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    # Processar imagens de treino (com_carro e sem_carro)
    for classe_nome, rotulo in [("com_carro", 1), ("sem_carro", 0)]:
        pasta_classe = pasta_carros / "treino" / classe_nome
        if not pasta_classe.exists():
            print(f"Aviso: Pasta de treino '{pasta_classe}' não encontrada.")
            continue

        pasta_saida_classe = pasta_saida / classe_nome
        pasta_saida_classe.mkdir(parents=True, exist_ok=True)

        for img_path in pasta_classe.iterdir():
            if img_path.is_file() and img_path.suffix.lower() in extensoes:
                nome_pgm = img_path.stem + ".pgm"
                caminho_pgm = pasta_saida_classe / nome_pgm

                if converter_para_pgm_16x16(img_path, caminho_pgm):
                    csv_linhas.append(f"{caminho_pgm},{rotulo}\n")

    # Salva o arquivo CSV de treino
    caminho_csv = pasta_saida / "treino.csv"
    with open(caminho_csv, "w") as f_csv:
        f_csv.writelines(csv_linhas)
    print(f"✅ Dataset de treino gerado em {pasta_saida} com {len(csv_linhas)} exemplos. CSV salvo em: {caminho_csv}")

    # Processar imagem de teste opcional passada no terceiro argumento
    if len(sys.argv) >= 4:
        img_teste = Path(sys.argv[3])
        if img_teste.exists():
            caminho_teste_pgm = pasta_saida / "teste.pgm"
            if converter_para_pgm_16x16(img_teste, caminho_teste_pgm):
                print(f"✅ Imagem de teste convertida com sucesso para: {caminho_teste_pgm}")
        else:
            print(f"❌ Imagem de teste especificada não existe: {img_teste}")

if __name__ == "__main__":
    main()

In [ ]:
# Executar a preparação das imagens de carro
!python /content/preparar_imagens_carros.py \
    /content/dataset_carros \
    /content/dataset_pgm \
    "/content/dataset_carros/teste/com_carro/Pasted image (23).png"

In [ ]:
# Executar o programa da rede neural com o dataset de treino e imagem de teste em PGM
!./rede_neural /content/dataset_pgm/treino.csv /content/dataset_pgm/teste.pgm

In [ ]:
# 1. Converter a nova imagem de teste para o formato PGM
!python /content/preparar_imagens_carros.py \
    /content/dataset_carros \
    /content/dataset_pgm \
    "/content/dataset_carros/teste/com_carro/Pasted image (24).png"

# Como o script salva por padrão como 'teste.pgm', vamos apenas rodar o classificador com ele
print("\n--- Executando a predição na nova imagem ---")
!./rede_neural /content/dataset_pgm/treino.csv /content/dataset_pgm/teste.pgm

In [ ]:
!ls /content/dataset_carros/teste/sem_carro

In [ ]:
!python /content/preparar_imagens_carros.py \
    /content/dataset_carros \
    /content/dataset_pgm \
    "/content/dataset_carros/teste/sem_carro/rua_vazia.png"

!/content/rede_neural \
    /content/dataset_pgm/treino.csv \
    /content/dataset_pgm/teste.pgm

In [ ]:
from pathlib import Path
import subprocess
import sys

pasta_teste = Path("/content/dataset_carros/teste/sem_carro")
extensoes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

imagens = sorted(
    arquivo for arquivo in pasta_teste.iterdir()
    if arquivo.is_file() and arquivo.suffix.lower() in extensoes
)

if not imagens:
    print(f"Não encontrei imagens em {pasta_teste}")
else:
    imagem = imagens[0]
    print(f"Imagem usada no teste: {imagem.name}")

    subprocess.run([
        sys.executable,
        "/content/preparar_imagens_carros.py",
        "/content/dataset_carros",
        "/content/dataset_pgm",
        str(imagem)
    ], check=True)

    subprocess.run([
        "/content/rede_neural",
        "/content/dataset_pgm/treino.csv",
        "/content/dataset_pgm/teste.pgm"
    ], check=True)

In [ ]:
pasta_teste = Path("/content/dataset_carros/teste/com_carro")

In [ ]:
from pathlib import Path
import subprocess
import sys

pasta_teste = Path("/content/dataset_carros/teste/com_carro")
extensoes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

imagens = sorted(
    arquivo for arquivo in pasta_teste.iterdir()
    if arquivo.is_file() and arquivo.suffix.lower() in extensoes
)

if not imagens:
    print(f"Não encontrei imagens em {pasta_teste}")
else:
    imagem = imagens[0]
    print(f"Imagem usada no teste: {imagem.name}")

    subprocess.run([
        sys.executable,
        "/content/preparar_imagens_carros.py",
        "/content/dataset_carros",
        "/content/dataset_pgm",
        str(imagem)
    ], check=True)

    subprocess.run([
        "/content/rede_neural",
        "/content/dataset_pgm/treino.csv",
        "/content/dataset_pgm/teste.pgm"
    ], check=True)

In [ ]:
from PIL import Image
from IPython.display import display

print(f"Imagem: {imagem.name}")
display(Image.open(imagem))

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

pasta = Path("/content/quadros_carros")
imagens = sorted(
    p for p in pasta.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

if not imagens:
    print(f"Não encontrei imagens em {pasta}")
else:
    imagem = imagens[0]
    foto = Image.open(imagem).convert("RGB")
    largura, altura = foto.size

    print(f"Arquivo: {imagem.name} | largura: {largura}px | altura: {altura}px")

    passo_x = max(1, largura // 10)
    passo_y = max(1, altura // 10)

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(foto)
    ax.set_xticks(range(0, largura, passo_x))
    ax.set_yticks(range(0, altura, passo_y))
    ax.grid(color="yellow", alpha=0.5)
    ax.set_title("Use os eixos para localizar os cantos do carro")
    plt.show()

In [ ]:
xmin = 10
ymin = 14
xmax = 244
ymax = 137

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display

foto = Image.open(imagem).convert("RGB")
desenho = ImageDraw.Draw(foto)

desenho.rectangle((10, 14, 244, 137), outline="red", width=3)
desenho.rectangle((10, 0, 55, 14), fill="red")
desenho.text((12, 1), "carro", fill="white")

display(foto)

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

caminho_imagem = Path(
    "/content/dataset_carros/teste/com_carro/carros3.png"
)

if not caminho_imagem.exists():
    print(f"Não encontrei a imagem: {caminho_imagem}")
else:
    foto = Image.open(caminho_imagem).convert("RGB")
    largura, altura = foto.size

    print(f"Arquivo: {caminho_imagem.name} | largura: {largura}px | altura: {altura}px")

    passo_x = max(1, largura // 10)
    passo_y = max(1, altura // 10)

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(foto)
    ax.set_xticks(range(0, largura, passo_x))
    ax.set_yticks(range(0, altura, passo_y))
    ax.grid(color="yellow", alpha=0.5)
    ax.set_title("Use os eixos para localizar os cantos do carro")
    plt.show()

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display

foto = Image.open("/content/dataset_carros/teste/com_carro/carros3.png").convert("RGB")
desenho = ImageDraw.Draw(foto)

desenho.rectangle((10, 14, 244, 137), outline="red", width=3)
desenho.rectangle((10, 0, 55, 14), fill="red")
desenho.text((12, 1), "carro", fill="white")

display(foto)

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

caminho_imagem = Path(
    "/content/dataset_carros/teste/com_carro/carro4.png"
)

if not caminho_imagem.is_file():
    print(f"Não encontrei a imagem: {caminho_imagem}")
else:
    foto = Image.open(caminho_imagem).convert("RGB")
    largura, altura = foto.size

    print(f"Imagem: {caminho_imagem.name} | {largura} × {altura} pixels")

    passo_x = max(1, largura // 10)
    passo_y = max(1, altura // 10)

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(foto)
    ax.set_xticks(range(0, largura, passo_x))
    ax.set_yticks(range(0, altura, passo_y))
    ax.grid(color="yellow", alpha=0.5)
    ax.set_title("Use os eixos para localizar os cantos do carro")
    plt.show()

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display

foto = Image.open("/content/dataset_carros/teste/com_carro/carro4.png").convert("RGB")
desenho = ImageDraw.Draw(foto)

desenho.rectangle((10, 14, 244, 137), outline="red", width=3)
desenho.rectangle((10, 0, 55, 14), fill="red")
desenho.text((12, 1), "carro", fill="white")

display(foto)